# Flight Price Prediction DataSet EDA and Feature engineering:----

### The objective of the study is to analyze the flight booking dataset obtained from the “Ease My Trip” website and to conduct various statistical hypothesis tests in order to get meaningful information from it.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as ply
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder

In [7]:
data = pd.read_excel('/Users/rkp/Downloads/Data_Train.xlsx')
data.to_csv('Data_Train.csv',index=False)
df=pd.DataFrame(pd.read_csv('Data_Train.csv'))

In [9]:
df.head(3)

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882


In [10]:
df.shape

(10683, 11)

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Airline          10683 non-null  str  
 1   Date_of_Journey  10683 non-null  str  
 2   Source           10683 non-null  str  
 3   Destination      10683 non-null  str  
 4   Route            10682 non-null  str  
 5   Dep_Time         10683 non-null  str  
 6   Arrival_Time     10683 non-null  str  
 7   Duration         10683 non-null  str  
 8   Total_Stops      10682 non-null  str  
 9   Additional_Info  10683 non-null  str  
 10  Price            10683 non-null  int64
dtypes: int64(1), str(10)
memory usage: 918.2 KB


In [12]:
df.describe()

,Price
count,10683.000000
mean,9087.064121
std,4611.359167
min,1759.000000
25%,5277.000000
50%,8372.000000
75%,12373.000000
max,79512.000000


In [13]:
df.isnull().sum()

Airline            0
Date_of_Journey    0
Source             0
Destination        0
Route              1
Dep_Time           0
Arrival_Time       0
Duration           0
Total_Stops        1
Additional_Info    0
Price              0
dtype: int64

In [14]:
df.dropna(axis=0,inplace=True)

IN dropna if we dont use inplace=True then it will remove the data for just same cell. It will not affect the dataframe

In [15]:
df.shape

(10682, 11)

In [16]:
df.isnull().sum()

Airline            0
Date_of_Journey    0
Source             0
Destination        0
Route              0
Dep_Time           0
Arrival_Time       0
Duration           0
Total_Stops        0
Additional_Info    0
Price              0
dtype: int64

In [18]:
df.duplicated().sum()

np.int64(220)

In [20]:
df.drop_duplicates(inplace=True)

In [21]:
df.shape

(10462, 11)

In [25]:
df.duplicated().sum()

np.int64(0)

In [26]:
df.head(1)

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897


### Extract Date, Month, Year from Date_of_Journey

In [27]:
df['date']=df['Date_of_Journey'].str.split('/').str[0]
df['month']=df['Date_of_Journey'].str.split('/').str[1]
df['year']=df['Date_of_Journey'].str.split('/').str[2]

### Convert extracted columns to integer type

In [28]:
df['date']=df['date'].astype(int)
df['month']=df['month'].astype(int)
df['year']=df['year'].astype(int)

### Drop original Date_of_Journey column

In [30]:
df.drop('Date_of_Journey',axis=1,inplace=True)

In [31]:
df.head(1)

,Airline,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,date,month,year
0,IndiGo,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897,24,3,2019


### Clean and Extract Arrival Hours and Minutes from Arrival_Time

In [35]:
df['Arrival_Time']=pd.to_datetime(df['Arrival_Time'],format='mixed')
# Extract components directly
df['arrival_hour']=df['Arrival_Time'].dt.hour
df['arrival_minute']=df['Arrival_Time'].dt.minute
df['arrival_date']=df['Arrival_Time'].dt.day

In [36]:
df.head(1)

,Airline,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,date,month,year,arrival_hour,arrival_minute,arrival_date
0,IndiGo,Banglore,New Delhi,BLR → DEL,22:20,2026-03-22 01:10:00,2h 50m,non-stop,No info,3897,24,3,2019,1,10,22


In [37]:
df.drop('Arrival_Time',axis=1,inplace=True)

In [38]:
df.head(1)

,Airline,Source,Destination,Route,Dep_Time,Duration,Total_Stops,Additional_Info,Price,date,month,year,arrival_hour,arrival_minute,arrival_date
0,IndiGo,Banglore,New Delhi,BLR → DEL,22:20,2h 50m,non-stop,No info,3897,24,3,2019,1,10,22


In [45]:
df.head(1)

,Airline,Source,Destination,Route,Duration,Total_Stops,Additional_Info,Price,date,month,year,arrival_hour,arrival_minute,arrival_date,dep_hour,dep_minute
0,IndiGo,Banglore,New Delhi,BLR → DEL,2h 50m,non-stop,No info,3897,24,3,2019,1,10,22,22,20


### One-Hot Encode Categorical Features

In [50]:
ohe=OneHotEncoder()
encoded_array=ohe.fit_transform(df[['Airline','Source','Destination']]).toarray()
encoded_df=pd.DataFrame(encoded_array,columns=ohe.get_feature_names_out(['Airline', 'Source', 'Destination']))
df_final = pd.concat([df, encoded_df], axis=1)
                                

## FINAL DATAFRAME

In [51]:
df_final.head(3)

,Airline,Source,Destination,Route,Duration,Total_Stops,Additional_Info,Price,date,month,...,Source_Chennai,Source_Delhi,Source_Kolkata,Source_Mumbai,Destination_Banglore,Destination_Cochin,Destination_Delhi,Destination_Hyderabad,Destination_Kolkata,Destination_New Delhi
0,IndiGo,Banglore,New Delhi,BLR → DEL,2h 50m,non-stop,No info,3897.0,24.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,7h 25m,2 stops,No info,7662.0,1.0,5.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,Jet Airways,Delhi,Cochin,DEL → LKO → BOM → COK,19h,2 stops,No info,13882.0,9.0,6.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
